<center> <strong> <span style="font-size: 33px;">03. Clasificación y chatbot con documentos jurídicos usando Retrieval Augmentend Generation (RAG) </span></strong> </center>





<span style = "font-size: 21px;"> 
Alejandro Villegas
</span>

<span style = "font-size: 21px;"> 
2025-06-26
</span>

Este proyecto/demostración está realizado en lenguaje Python. Para su construcción se utilizó el entorno de desarrollo Spyder y para su presentación en este documento se utilizó el formato Jupyter Notebook.

En el repositorio de Github de este proyecto encontrarás, los insumos y el código y los productos del código aquí explicados. Esto para facilitar su reproductibilidad <a href="https://github.com/AleVillegas9/03_RAG_CLAS_CHAT" target="_blank">Repositorio Github</a>

**Nota:** Debido a la naturaleza de los insumos, la seguridad y privacidad deben ser cuidadas. Por lo cual, a parte de las medidas obvias (como ocultar contraseñas y APIKEY's, se omitirán los resultados de los códigos. Sumado a ello, las direcciones de carpetas, contenidos de variables y documentos privados será omitido. Finalmente, si bien la aplicación está montad en *streamlit* por políticas de la plataforma, cada cierto tiempo de inactividad la aplicación queda deshabilitada. Si usted desea probar la demo de la aplicación puede ponerse en contacto conmigo en el siguiente correo: java.law.1993@gmail.com

<center> <strong> <span style="font-size: 24px;"> Problema a resolver </span></strong> </center>

En la era de la información es muy común que, por un lado se produzca una mayor cantidad de información, pero por otro lado, nosotros cada vez contamos con menos tiempo para analizarlos. Por lo cual, sería muy útil tener herramientas que automatizarán el proceso de *leer* la información por nosotros, y *procesarla*. Por suerte, también vivimos en una época en donde se puede utilizar la inteligencia artificial, específicamente los *Large Language Models* (LLM'S) para ayudarnos con la tarea. 

En este sentido, en este *demo* se abordarán el cómo se puede usar los LLM's para abordar dos problemas, a saber ¿cómo se puede automatizar la clasificación de distintos textos? y ¿Cómo se puede facilitar la *exploración* de la información contenida en los textos?. Para resolver estas preguntas se creará una aplicación con doble funcionalidad. Por un lado, clasificará distintos documentos normativos, y por otro lado, nos permitirá seleccionar el texto normativo que querramos y *chatear* con él. 

Para la realización de este *demo* se utilizó la normativa del Consejo de la Judicatura Federal, la cual puede ser consultada en https://apps.cjf.gob.mx/normativa/Index


<center> <strong> <span style="font-size: 24px;"> Insumos </span></strong> </center>

Las herramientas que se necesitan para llevar a cabo este proyecto son:

- Python
- Acceso a un LLM mediante una API.
- Streamlit
- Github
- Una *vector store database*, en este caso Pinecone.
- Librería / Framework Langchain. 

Los insumos necesarios para ejecutar este proyecto son:

- Los documento a clasificar/explorar (en este caso la normativa del Consejo de la Judicatura).
- Las categorias sobre las que se hará la clasificación.

Se presupone que el usuario cuenta con los documentos a análizar en formato PDF.  En caso de que se tenga en otro formato, como por ejemplo word, se presupone la transformación de los documentos a formato PDF. Esto se puede realizar usando páginas como ilovepdf, o si se requiere hacer la transformación de manera masiva y automatizada, se puede hacer utilizando la librería de Python docx2pdf

Otro presupuesto es que los PDF's se encuentren en un formato tal que permitan la extracción de texto, es decir, que el contenido del documento no sea una imagen. En caso de que lo sea, se necesita convertirlo a texto o a un formato que permmita la extracción del texto. Para lograr esto se puede usar la tecnología *Optical Character Recognition* (OCR). Para hacer uso de ella de manera automatizada se puede usar dede la librería de Python pytesseract, aunque se necesita tener instalado Tesseract OCR. 

Finalmente, se presupone que todos los documentos (las normas en este caso) están guardadas en una sola carpeta. 

<center> <strong> <span style="font-size: 27px;"> Metodología </span></strong> </center>

<strong> <span style="font-size: 22px;">PARTE A: Preparación de insumos </span></strong>

El objetivo principal de la parte A, es preparar el contenido de los documentos para que se pueda usar el *Retrieval Augmented Generation* (RAG), y con ello se pueda clasificar y explorar los documentos de manera eficiente. Para una explicación de cómo funciona el RAG se puede consultar: https://aws.amazon.com/es/what-is/retrieval-augmented-generation/

<strong> <span style="font-size: 20px;">Paso 1: Carga de los documentos </span></strong>

Suponiendo que los ducumentos con los que se dese trabajar se encuentrana en formato PDF y que el contenido sea texto y no imagen, comenzamos el proceso extrayendo su contenido y guardandolo en una *lista*. En donde cada elemento es un objeto *string* del contenido de cada documento. 

In [ ]:
#Primero las librerias, por supuesto
import os
import PyPDF2

#Después creamos la función que extraiga la info y la guarde en una lista, pdf_text
def extract_text_from_pdfs(pdf_folder_path):
    pdf_texts = {}
    for pdf_file in os.listdir(pdf_folder_path):
        if pdf_file.endswith('.pdf'):
            with open(os.path.join(pdf_folder_path, pdf_file), 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                text = ''
                for page in reader.pages:
                    text += page.extract_text()
                pdf_texts[pdf_file] = text
    return pdf_texts

# Establecemos el path de la carpeta con los pdf's a extraer

pdf_folder_path = (r"direccion de tu carpeta")

#usamos la función y ponemos que la lista se llama normas base

normas_bae = extract_text_from_pdfs(pdf_folder_path)

<strong> <span style="font-size: 22px;"> Paso 2: Fragmentación de los contenidos </span></strong>

Uno de los procesos más comunes para trabajar con LLM's y documentos muy grandes es el proceso de framgentación (*chunking*) de los contenidos a análizar. Como su nombre lo indica, este proceso consiste en dividir el texto en fragmentos que el LLM o *modelo* pueda reconocer e interpretar. Así pues, fragmentamos cada uno de los contenidos. Para ello, usaremos funciones de la librería langchain. 

In [ ]:
# Como siempre, importamos la librería a utilizar.

from langchain.text_splitter import CharacterTextSplitter

#Hacemos el fragmentador. Establecemos que el separador sea un espacio en blanco para que separe por palabras. 
text_splitter = CharacterTextSplitter(separator = ' ', chunk_size = 2500, chunk_overlap = 50) 

#Hacemos un diccionario donde guardar los fragmentos. La llave del diciconario será el nombre de la norma/documento + el número de fragmento.
normas_fragmentadas = {} 

#iteramos el fragmentador para cada norma
for norma, contenido in normas_base.items(): 
    fragmentos = text_splitter.split_text(contenido) 
    normas_fragmentadas[norma] = fragmentos

<strong> <span style="font-size: 22px;">Paso 3: Vectorización (*embedding*) de los fragmentos. </span></strong>

Como el nombre del paso lo indica, se convierte cada uno de los fragmentos (*chunks*) en vectores numéricos. Cada uno de los elementos del vector está construido de tal manera que recupera el significado semántico de los elementos que representan. De tal manera que dos elementos semánticamente cercanos, tendrán un vector similar. Este paso es importante, pues permite hacer busquedas más eficientes, y el uso de bases de datos vectoriales, lo cual es un paso necesario paraa el uso del RAG. 

Para este demo, e utilizó el modelo "text-embeddig-ada-002" de OPENAI.

In [ ]:
# Lo de siempre, cargar librerias

from openai import OpenAI

#Nos conectamos con la API

usuario = OpenAi(api_key = "Obvio esto es secreto :p)

#creamos la funcion vectorizadora
def get_embedding (text, model = "text-embedding-ada-002"):
    text = text.replace('\n', " ")
    return usuario.embeddings.create(input = [text], model = model). data[0].embedding

#Iteramos la función en cada fragmento y lo guardamos en la lista data []

for doc_id, chunks in normas_fragmentadas.items():
    for i, chunk in enumerate(chunks):
        chunk_id = f"{doc_id}_chunk_{i}"
        vector = get_embedding(chunk)  
        data.append((chunk_id, vector))

#Nota importante: Como Pinecone no nos permitirá subir tantos fragmentos, es necesario dividir la lista data [] en sublistas más pequeñas

data1 = data[0:400]
data2=data[400:800]
data3=data [800:1200]
data4 = data [1200:1600]
data5 =data[1600:1895]

#De 400 en 400 se podrán subir sin problema. 
                 

<strong> <span style="font-size: 22px;"> Paso 4: Guardar los vectores en un una base de vectores (*pinecone*) </span></strong>

Para que se puedan consultar los diferentes vectores y de esa manera aplicar el RAG, se deben de guardar los vectores de los documentos en una base vectorial (*index*), en este caso Pinecone. Naturalmente, para ello, ya debemos contar con una cuenta y una APIKey que nos permita acceder y usar la plataforma. La creación del *index* donde se guardarán los fragmentos vectorizados, se puede hacer desde Python. 

Se recomienda que cada *index* tenga un sólo propósito, entonces como aquí la aplicación tendrá dos objetivos (clasificar y explorar), se harán dos *index*. El primero que crearemos será el de clasificación. 

In [ ]:
# Lo de siempre... librerias
from pinecone import Pincenoce, ServerlessSpec

# Establecemos la apikey
pc = PinceCcone(api_key = "Obvio esto también es secreto  :p", enviroment = "us-west1-gcp")

# Cremos el index

pc.create_index(
    name="s3indexfuentes", #O ponle el nombre que quieras
    dimension=1536, # este deve ser del mismo tamanio del modelo de embedding que uso. fijarse en eso. 
    metric="euclidean", # Remplazar con la metrica que queremos. Esta es la más común pero hay otras. Prueba y error. 
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
) 


Una vez creado el index, lo inciamos y subimos cada subconjunto de datos. 

In [ ]:
#iniciamos el index en python

s3indexfuentes = pc.Index("s3indexfuentes")

#subimos cada subconjunto de los fragmentos vectorizados

s3indexfuentes.upsert(vectors = data5) #iterar sobre todos los subcojuntos 


En la plataforma Pinecone esto se debería ver algo así:

<div style="text-align: center;">
  <p><strong>Imagen 1:  Index Pinecone</strong></p>
  <img src="imagenes/01_pinecone.png" width="70%">
  <p><em>Fuente: Elaboración propia</em></p>
</div>

<strong> <span style="font-size: 22px;"> Paso 5: Preparación de los metadatos de los vectores  </span></strong>

A cada uno de los vectores guardados en Pinecone, se les puede anexar metadatos. Como su nombre lo indica, en este apartado se pueden anexar datos como el titulo, de los documentos, autor, o incluso, como en este caso, el contenido original del texto que representa el vector en cuestion. Esto es importante, porque cuando el LLM "llame" al vector, esta llamada puee recuperar ya sea el contenido numérico del vector, o el contenido en lenguaje natural del fragmento, lo último es lo que importa, y solo se recuperará si dicho contenido se encuentra en los metadatos. 

En este caso, a cada fragmento/vector, se le asociará el nombre de la norma a la que pertenece, el link de la misma, y un resumen de ella. Para hac resto, crearé un dicionario con estos datos, donde la llave será el nombre distintivo de cada norma/documento original. de esa manera, como los fragmentos/vectores, empiezan con el mismo nombre, se podrá asociar.En este caso, como se va a clasificar, quiero que la respuesta que me de, contenga los datos antes descritos. 

In [ ]:
#Creamos la lista de asociaciones, (por fin no se empeiza importando librerias)

asociaciones_fuente ={
    '2013-3-0-A':'Nombre de la Norma: Acuerdo general 32013 relativo a la determinación del número y límites territoriales de los circuitos judiciales. Link:https://apps.cjf.gob.mx/normativa/Recursos/2013-3-0-AC_V331.PDF Resumen:El documento esboza una serie de acuerdos y reformas emitidos por el Consejo de la Judicatura Federal en México, detallando la reestructuración y especialización de los órganos judiciales en diferentes regiones del país. Los acuerdos abordan la establecimiento de nuevos tribunales, cambios en la jurisdicción y los límites territoriales, la infraestructura y el equipo necesarios para los tribunales y las disposiciones transitorias para la implementación de estos cambios. El objetivo es mejorar la eficiencia y efectividad del sistema judicial en México a través de una mejor coordinación y especialización.',
    '2013-40-2-':'Nombre de la Norma: Acuerdo que reglamenta la organización y funcionamiento del CFJ. Link: https://apps.cjf.gob.mx/normativa/Recursos/2013-40-2-AC_V73.PDF Resumen: El documento emitido por el Consejo de la Judicatura Federal en 2013 describe la organización, funciones y reformas dentro de la Judicatura Federal de México. Se centra en modernizar las normas institucionales, aclarar los procedimientos y garantizar la eficiencia y seguridad jurídica. El documento detalla las responsabilidades y funciones de varios departamentos dentro del Poder Judicial, incluida la supervisión de asuntos judiciales, la prestación de apoyo legal, la administración de finanzas, la implementación de políticas anticorrupción y coordinar los esfuerzos de comunicación. También describe los acuerdos realizados por el Consejo entre 2013 y 2023, abordando reformas y modificaciones en diversos aspectos de la organización y el funcionamiento del Consejo.',
    '2014-36-0-':'Nombre de la Norma: Acuerdo General que regula los Centros de Justicia Penal Federal. Link:https://apps.cjf.gob.mx/normativa/Recursos/2014-36-0-AC_V18.PDF Resumen: El Acuerdo General 36/2014 del Pleno del Consejo de la Judicatura Federal regula los Centros Federales de Justicia Penal, delineando la organización y funcionamiento de estos centros dentro del ordenamiento jurídico mexicano. Establece los roles y responsabilidades de diversos cargos judiciales, incluido el Administrador del Centro de Justicia, y detalla los requisitos y deberes de este cargo. El documento también discute la derogación y modificación de diversos artículos relacionados con la procedimientos, el establecimiento de centros judiciales y el funcionamiento de estos centros. Además, describe los acuerdos y regulaciones con respecto a la selección y permanencia de los administradores, la organización y el funcionamiento de los recursos Tribunales Colegiados, y el funcionamiento del Tribunal Colegiado de Apelación. El documento ha sufrido varias reformas y enmiendas a lo largo de los años, con el acuerdo más reciente, General 32/2023, centrado en la programación de audiencias en Centros de Justicia Penal.',
    '2014-56-1-':'Nombre de la Norma:Acuerdo sobre actividad administrativa del CJF. Link:https://apps.cjf.gob.mx/normativa/Recursos/2014-56-1-AC_V81.PDF Resumen: El Consejo de la Judicatura Federal emitió un acuerdo integral que describe las disposiciones administrativas para sus actividades, enfatizando la independencia, simplificando los procesos regulatorios y abordando diversos aspectos como los recursos humanos, la presupuestación, procedimientos de adquisición y gobernanza digital. El acuerdo contempla normas para el uso de firmas electrónicas, herramientas digitales y el manejo de archivos personales para servidores públicos, así como lineamientos para el uso de códigos QR, reserva fondos y procesos de adquisición. También detalla los procedimientos para administrar los fondos de ahorro, proporcionar servicios de cuidado infantil y apoyo de vivienda para los funcionarios públicos, al tiempo que enfatiza el cumplimiento de las leyes, regulaciones y el bienestar de los niños y servidores públicos. Además, el documento describe los procedimientos para arrendar, adquirir, administrar y enajenar propiedades inmobiliarias, la gestión financiera y los requisitos de presentación de informes dentro del Consejo de la Judicatura Federal. También discute acuerdos relacionados con la protección civil, los nombramientos judiciales y la reestructuración organizativa, con un enfoque en mejorar la transparencia, la rendición de cuentas y la eficiencia dentro del Consejo.',
    '2014-56-2-':'Nombre de la Norma: Acuerdo sobre actividad administrativa de los órganos jurisdiccionales. Link:https://apps.cjf.gob.mx/normativa/Recursos/2014-56-2-AC_V37.PDF Resumen: El documento emitido por el Pleno del Consejo de la Judicatura Federal en México establece normas para las actividades administrativas dentro de los órganos judiciales, que abarcan las horas de trabajo, los períodos de vacaciones, el mantenimiento de registros electrónicos, los sistemas de gestión de casos y las funciones de coordinadores administrativos y unidades notariales. Su objetivo es agilizar los procesos administrativos, garantizar la eficiencia, aclarar los procedimientos y enfatizar la importancia del mantenimiento de registros precisos, la conducta ética y el cumplimiento de pautas para un mejor control de los casos y procesos legales. También describe los procedimientos para manejar asuntos urgentes, el funcionamiento de las oficinas comunes de correspondencia, la creación de Unidades de Notificación Judicial y el nombramiento y los requisitos para personal, además, aborda el registro y gestión de procesos judiciales, el uso de sistemas de gestión judicial, la publicación de listados judiciales e información estadística, y la digitalización de expedientes judiciales para mejorar el acceso a justicia y eficiencia en los procesos legales. El documento también cubre diversos procedimientos legales, acuerdos y reformas realizadas por el Pleno, incluidas órdenes judiciales, apelaciones, actividades administrativas, reestructuración de órganos judiciales, implementación de nuevos sistemas, y el uso de la videoconferencia en el sistema judicial, con el objetivo de mejorar la eficiencia, la coordinación y el acceso a la justicia dentro del sistema judicial mexicano.',
    '2018-28-2-':'Nombre de la Norma: Acuerdo en materia de protección de datos personales del CJF. Link:https://apps.cjf.gob.mx/normativa/Recursos/2018-28-2-AC_V03.PDF Resumen: El Consejo de la Judicatura Federal emitió un acuerdo general sobre la protección de datos personales, que describe los procedimientos para manejar las solicitudes y garantizar el cumplimiento de las leyes de protección de datos. Discute los principios para proteger los datos personales, el consentimiento para el procesamiento de datos y medidas para la seguridad de los datos. El documento también establece responsabilidades para los servidores públicos en el manejo de datos personales y describe los procedimientos para ejercer los derechos ARCO. Adicionalmente, detalla la implementación de la mano de obra reformas de justicia, incluida la creación de Tribunales Federales de Trabajo en varios estados de México.',
    '2018-45-1-':'Nombre de la Norma: Acuerdo en materia de responsabilidades administrativas, situación patrimonial, control y rendición de cuentas. Link:https://apps.cjf.gob.mx/normativa/Recursos/2018-45-1-AC_V12.PDF Resumen: El Acuerdo General del Pleno del Consejo de la Judicatura Federal establece disposiciones para las responsabilidades administrativas, el control financiero y la rendición de cuentas dentro del poder judicial. Describe la autoridad del Consejo para supervisar la judicial, emitir acuerdos generales, regular investigaciones y sanciones por responsabilidades administrativas, y enfatizar principios como la legalidad, la presunción de inocencia y el respeto a los derechos humanos. El documento detalla los procedimientos para investigar y sancionar las responsabilidades administrativas, incluida la creación de una Unidad de Investigación de Responsabilidades Administrativas, declaración de bienes e intereses, responsabilidades de los servidores públicos, imposición de sanciones, notificaciones, medidas cautelares, reembolso de ingresos perdidos, sanciones económicas, monitoreo de declaraciones de activos y manejo de procedimientos de responsabilidad administrativa. También enfatiza la transparencia, la rendición de cuentas y la importancia de incorporar la perspectiva de género en los casos de conducta sexual inapropiada o violencia de género. Además, el documento describe los procedimientos y regulaciones para la responsabilidad administrativa, incluida la presentación de apelaciones, reconsideraciones, quejas contra decisiones, violaciones, medidas para restablecer el status quo, aplicación de sanciones, sistema de justicia en línea, evaluaciones financieras de servidores públicos, inspección de órganos judiciales, entrega y recepción de materiales administrativos, conciliación de saldos bancarios, lineamientos para investigaciones administrativas, acuerdos relacionados con la Reforma Laboral de Justicia, creación de Juzgados Federales de Trabajo e incorporación de perspectiva de género en los procesos disciplinarios judiciales.',
    '2019-9-2-A':'Nombre de la Norma: Acuerdo en materia de transparencia y acceso a la información pública del CJF. Link:https://apps.cjf.gob.mx/normativa/Recursos/2019-9-2-AC_V05.PDF Resumen: El Acuerdo General del Pleno del Consejo de la Judicatura Federal establece disposiciones para la transparencia y el acceso a la información pública dentro del Consejo, enfatizando la libertad de expresión y el acceso a la información. Describe los roles y responsabilidades, define términos clave y establece pautas para manejar las solicitudes de información pública. El documento también detalla los procedimientos para manejar solicitudes, publicar versiones públicas de documentos, acceder a decisiones judiciales y implementar reformas en la justicia laboral. Adicionalmente, brinda información sobre Juzgados Federales del Trabajo en diferentes estados de México y acuerdos relacionados con la difusión de videograbaciones de sesiones públicas.',
    '2020-3-1-A':'Nombre de la Norma: Acuerdo que establece la organización y conservación de los archivos administrativos del CJF. Link:https://apps.cjf.gob.mx/normativa/Recursos/2020-3-1-AC_V02.PDF Resumen: El Consejo de la Judicatura Federal emitió un acuerdo general que describe la organización y preservación de los archivos administrativos, basado en las leyes mexicanas y destinado a garantizar el acceso a la información y la preservación de documentos. procedimientos para la gestión de archivos, incluida la destrucción, transferencia, organización, conservación y eliminación de documentos. También enfatiza la importancia de una adecuada gestión documental, clasificación y cumplimiento de las normas de archivo. acuerdo establece responsabilidades para administrar archivos, describe los procedimientos para manejar la correspondencia oficial y detalla la formación y el funcionamiento de un grupo interdisciplinario para ayudar a determinar los valores de los documentos y adicionalmente, aborda la importancia del Archivo Histórico del Poder Judicial de la Federación, los procedimientos para solicitar informes de valoración de documentos y la formación de un Manual de Archivos Institucionales. El acuerdo fue aprobado por el Consejo el 19 de febrero de 2020, y publicado para una difusión más amplia en marzo de 2020.',
    '2020-12-0-':'Nombre de la Norma: Acuerdo que regula la integración y trámite de expediente electrónico y el uso de videoconferencias en los órganos jurisdiccionales. Link:https://apps.cjf.gob.mx/normativa/Recursos/2020-12-0-AC_V07.PDF Resumen: El Acuerdo General 12/2020 del Pleno del Consejo de la Judicatura Federal en México se centra en regular el uso de archivos electrónicos y videoconferencias en materia judicial para mejorar el acceso a la justicia, particularmente durante la COVID-19 pandemia. Permite el uso de medios electrónicos en los procedimientos judiciales, describe las pautas para las notificaciones electrónicas, las firmas digitales y las videoconferencias, y enfatiza la importancia de mantener la integridad y la seguridad de archivos electrónicos. El documento tiene como objetivo modernizar y agilizar el proceso legal a través de medios digitales al tiempo que garantiza la seguridad y validez de los documentos electrónicos. También detalla los esfuerzos de modernización del Consejo de la Judicatura Federal, incluida la implementación de la tecnología de videoconferencia, y enfatiza la importancia de la seguridad, el cifrado y el respeto de los derechos de todas las partes involucradas en los procedimientos legales.',
    '2021-0-134':'Nombre de la Norma: Acuerdo que reglamenta la Carrera Judicial. Link:https://apps.cjf.gob.mx/normativa/Recursos/2021-0-134-DD_V16.PDF Resumen: El documento emitido por el Consejo de la Judicatura Federal en México el 3 de noviembre de 2021, introduce nuevas normas y reglamentos para la Carrera Judicial destinados a combatir el nepotismo, promover la igualdad de género y mejorar la profesionalidad dentro de la las reformas se centran en establecer un sistema basado en el mérito, mejorar las oportunidades de capacitación y abordar la corrupción y el nepotismo. Describe los requisitos, procesos y principios para el ingreso, la promoción y la evaluación en el Poder Judicial Carrera, así como procedimientos para concursos judiciales, revisiones administrativas y acciones disciplinarias. El documento también cubre asignaciones de trabajo, ratificación de jueces, manejo de conflictos de intereses, evaluaciones de desempeño, acoso sexual, género la igualdad y el desarrollo profesional de los funcionarios judiciales, enfatizando la transparencia, la eficiencia y el cumplimiento de los criterios establecidos en la administración de justicia.',
    '2023-6-2-A':'Nombre de la Norma: Acuerdo en materia de valoración, destrucción, digitalización, transferencia, resguardo y destino final de los expedientes judiciales. Link:https://apps.cjf.gob.mx/normativa/Recursos/2023-6-2-AC_V01.PDF Resumen: El Consejo de la Judicatura Federal emitió un acuerdo general que describe los procedimientos para la gestión, preservación y destrucción de los registros judiciales en México. El acuerdo simplifica las actividades al eliminar la categoría de valoración y digitalizar documentos físicos para almacenamiento electrónico. Establece pautas para la clasificación, transferencia y destrucción de expedientes judiciales en función de su relevancia y antigüedad. El documento enfatiza la importancia del cumplimiento de las leyes de protección de datos, el resguardo de archivos con trascendencia jurídica, y la coordinación de archivos dentro del Poder Judicial de la Federación. El acuerdo tiene como objetivo agilizar los procesos documentales, optimizar los recursos y garantizar la transparencia y el acceso a la información dentro del Consejo de la Judicatura Federal.'}

In [ ]:
# Sacmos el nombre de cada documento, y hacemos una tupla con el nombre, y los datos

for tupla in data:
    nombre = tupla[0]
    nombres.append(nombre)

#hacemos la lista de cada fuente. 

for nombre in nombres:
    prefijo = nombre[:10]  #esta linea me dira cual sera el criterio para clasificar el nombre asociado, en este caso que coincidan en los primeros 10 caracteres.
        lista_fuentes.append(asociaciones_fuente[prefijo])

# creamos el diccionario con las fuentes y los nombres de los fragmentos asociados. 
metadatos_fuentes ={nombres:value for nombres, value in zip(nombres,lista_fuentes)}

# Actualizamos el index en pinecone

for key, value in metadatos_fuentes.items():
    s3indexfuentes.update(id = key, set_metadata = {'fuente': f'{value}' }) 

Nuestros vectores guardados en el *index* ya tienen metadatos. Son los que están en el rectangulo negro. 

<div style="text-align: center;">
  <p><strong>Imagen 2:  Index Pinecone con metadatos</strong></p>
  <img src="imagenes/02_metadatos.png" width="70%">
  <p><em>Fuente: Elaboración propia</em></p>
</div>

Para la creación del *index* para explorar, se sigen los mismos pasos, sólo que en al actualizar los metadatos, se sube el contenido original de los fragmentos. Pero es el mismo proceso, desde la creación del index. 

En este *index* los metadatos se verán así:

<div style="text-align: center;">
  <p><strong>Imagen 3:  Index Pinecone metadatos del index para chatbot</strong></p>
  <img src="imagenes/03_metadados2.png" width="70%">
  <p><em>Fuente: Elaboración propia</em></p>
</div>

Notece que los metadados difieren en ambos *index*. Una vez con estos preparados podemos avanzar. 

<strong> <span style="font-size: 22px;"> Parte B: Montado en streamlit </span></strong>

*Streamlit* es una pltaforma muy fácil de usar que permite montar aplicaciones de nuestros proyectos. Se integra muy bien en Python, y es gratuita, sólo necesitas una cuenta en Github. Presuponemos que el usuario ya cuenta con esto. 
La manera que funciona la plataforma, es que creamos la aplicación desde un código en Python, guardamos el código en un repositorio de Github, entramos a la página de *streamlit*, y creamos la aplicación. Así, pues, veamos como se puede crear esta app.

Nota: No es necesario subir al repositorio el código de la parte A, sólo el de la B. 

<strong> <span style="font-size: 22px;"> Paso 6: Datos introductorios </span></strong>

Una vez preparados los datos es hora de montar la aplicación. Para ello, se utilizará la plataforma *streamlit*. Para ello, empezamos montando los elementos del encabezado.

In [ ]:
#Ahora no nos salvamos de importar librerias

import streamlit as st

#Ponemos los datos generales, como título etc. 

st.title('Asistente jurídico Aymr') #Cambiar el nombre cuando se decida uno

st.image("judicatura.jpg", use_container_width=True) #Quizá también añadir un logotipo del proyecto. Que lo haga la IA xd. 


st.subheader("Aymr es un asistente jurídico, impulsado con inteligencia artificial, que te ayudará en la busqueda de normas que sean relevantes para tu caso. Además de que te permitirá interactuar con dichas normas. Su funcionamiento es muy sencillo. Primero debes de seleccionar la categoria en la que se enmarca tu caso en cuestión. Aymr te dará una lista de las normas con una mayor posibilidad de relacionarse con tu caso. Segundo, elije una norma con la que quieras chatear, y hazle preguntas")


st.subheader(" ADVERTENCIA: Esta herramienta es sólo de apoyo para la labor de los abogados. DE NINGUNA MANERA SUSTITUYE LA LABOR DE INVESTIGACIÓN Y AUTONOMÍA DEL ABOGADO")


st.header("Clasificador")

st.subheader ("Instrucciones: A continuación se te presenta una lista de distintas categorias en las que se clasifican las normas jurídicas. Por favor, selecciona la categoria de la cual te gustaría conocer los textos relacionados")


Esto, al iniciar la app, se vería así. 

<div style="text-align: center;">
  <p><strong>Imagen 4: Encabezado de la app</strong></p>
  <img src="imagenes/04_encabezado.png" width="70%">
  <p><em>Fuente: Elaboración propia</em></p>
</div>

<strong> <span style="font-size: 22px;">Paso 7: Clasificación.</span></strong>

Una vez ya preparado los documentos y los vectores, es momento de preparar nuestro LLM para que nos de respuesta o clasifique. Para ello, primero se carga el modelo, y posteriormente se crea el *prompt* adecuado para que nos clasifique los documentos. En este sentido, se deben indicar las categorias por las cuales se desea clasificar, además, se recomienda dar ejemplos y explicaciones de dichas categorias. El procedimiento general, es que, nuestro *prompt* se convertirá en un vector, después, dicho vector, será utilizado para una busqueda semátca o de proximidad con los vectores guardados en pinecone. Así se recuperarán los vectores más relevantes, cuyo contenido se utilizará como contexto para la respuesta que nos dará el modelo. 

En este caso, queremos clasificar si las normas son normativiad relevante para las áreas administrativas, para los órganos jurisdicionales o para los acuerdo organizacionales del Consejo de la Judicatura Federal.

Como los resultados de la clasificación serán fijos, se ahorraron recursos haciendo la clasificación en un *script* aparte, y pasando los resultados a la aplicación. Así pues, para clasificar lo que se hace es, iniciar el *index* como en los pasos anteriores. y despues:

In [ ]:
# Lo de siempre
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

#Primero, creamos el embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", openai_api_key= 'SECRETO')


#Despues Hacemos el vectorstore
vector_store_clasificador = PineconeVectorStore(index = s3indexfuentes, embedding= embeddings, text_key= 'fuente') 
# Hacemos la función de clasifiación.

retrieverprop = vector_store_clasificador.as_retriever(search_type = 'similarity_score_threshold', search_kwargs = {"score_threshold": 0.50}) 
#Aqui puedo manejar el porcentaje de similitud que servira para saber que textos me dara.

#Hacemos las query, una por categoria. Podemos complejizarlas para obtener mejores resultados. 

query1 = '¿Qué normativa son Acuerdos Organizacionales del Consejo de la Judicatura federal?'  

query2 =  '¿Qué normativa pertenece a la Normatividad relevante para Órganos Jurisdiccionales?' 

query3 =  '¿Qué normativa normativa refiere a la Nomatividad relevante para Áreas Administrativas?'

#pregfuntamos y recuperamos la respuesta por cada query

respuestaprop1 = retrieverprop.invoke(query1)

print(respuestaprop1)


Notece, que en este paso se aplica algo importante **el uso de un LLM para resumir textos extensos** Esto llevado a nuestra apicación sería algo como lo siguiente. Notece que las respuestas que nos dio el código anterior, son lo que se usaron para construir esta parte de la app.

In [ ]:
#Ponemos el select box. 

categoria_usur = st.selectbox("Elije la categoria que enmarca tu caso: ", ['Acuerdos Organizacionales del Consejo de la Judicatura Federal', 'Normatividad Relevante para Órganos Jurisdiccionales', 'Nomatividad Relevante para Áreas Administrativas']) 

# Aplicamos la clasificacion

if st.button ('Clasificar'):
    st.markdown ('Las normas relevantes para la categoria seleccionada son:')
    if categoria_usur == 'Acuerdos Organizacionales del Consejo de la Judicatura Federal':
        st.write('1. Nombre de la Norma: 2014-56-1-AC_V80\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2014-56-1-AC_V80.PDF\n Resumen: El Consejo de la Judicatura Federal emitió un acuerdo integral que describe las disposiciones administrativas para sus actividades, enfatizando la independencia, simplificando los procesos regulatorios y abordando diversos aspectos como los recursos humanos, la presupuestación, procedimientos de adquisición y gobernanza digital. El acuerdo contempla normas para el uso de firmas electrónicas, herramientas digitales y el manejo de archivos personales para servidores públicos, así como lineamientos para el uso de códigos QR, reserva fondos y procesos de adquisición. También detalla los procedimientos para administrar los fondos de ahorro, proporcionar servicios de cuidado infantil y apoyo de vivienda para los funcionarios públicos, al tiempo que enfatiza el cumplimiento de las leyes, regulaciones y el bienestar de los niños y servidores públicos. Además, el documento describe los procedimientos para arrendar, adquirir, administrar y enajenar propiedades inmobiliarias, la gestión financiera y los requisitos de presentación de informes dentro del Consejo de la Judicatura Federal. También discute acuerdos relacionados con la protección civil, los nombramientos judiciales y la reestructuración organizativa, con un enfoque en mejorar la transparencia, la rendición de cuentas y la eficiencia dentro del Consejo.\n\n2. Nombre de la Norma: 2013-40-2-AC_V72\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2013-40-2-AC_V72.PDF\n Resumen: El documento emitido por el Consejo de la Judicatura Federal en 2013 describe la organización, funciones y reformas dentro de la Judicatura Federal de México. Se centra en modernizar las normas institucionales, aclarar los procedimientos y garantizar la eficiencia y seguridad jurídica. El documento detalla las responsabilidades y funciones de varios departamentos dentro del Poder Judicial, incluida la supervisión de asuntos judiciales, la prestación de apoyo legal, la administración de finanzas, la implementación de políticas anticorrupción y coordinar los esfuerzos de comunicación. También describe los acuerdos realizados por el Consejo entre 2013 y 2023, abordando reformas y modificaciones en diversos aspectos de la organización y el funcionamiento del Consejo.\n\n3. Nombre de la Norma: 2018-45-1-AC_V12\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2018-45-1-AC_V12.PDF\n Resumen: El Acuerdo General del Pleno del Consejo de la Judicatura Federal establece disposiciones para las responsabilidades administrativas, el control financiero y la rendición de cuentas dentro del poder judicial. Describe la autoridad del Consejo para supervisar la judicial, emitir acuerdos generales, regular investigaciones y sanciones por responsabilidades administrativas, y enfatizar principios como la legalidad, la presunción de inocencia y el respeto a los derechos humanos. El documento detalla los procedimientos para investigar y sancionar las responsabilidades administrativas, incluida la creación de una Unidad de Investigación de Responsabilidades Administrativas, declaración de bienes e intereses, responsabilidades de los servidores públicos, imposición de sanciones, notificaciones, medidas cautelares, reembolso de ingresos perdidos, sanciones económicas, monitoreo de declaraciones de activos y manejo de procedimientos de responsabilidad administrativa. También enfatiza la transparencia, la rendición de cuentas y la importancia de incorporar la perspectiva de género en los casos de conducta sexual inapropiada o violencia de género. Además, el documento describe los procedimientos y regulaciones para la responsabilidad administrativa, incluida la presentación de apelaciones, reconsideraciones, quejas contra decisiones, violaciones, medidas para restablecer el status quo, aplicación de sanciones, sistema de justicia en línea, evaluaciones financieras de servidores públicos, inspección de órganos judiciales, entrega y recepción de materiales administrativos, conciliación de saldos bancarios, lineamientos para investigaciones administrativas, acuerdos relacionados con la Reforma Laboral de Justicia, creación de Juzgados Federales de Trabajo e incorporación de perspectiva de género en los procesos disciplinarios judiciales.')
    elif categoria_usur == 'Normatividad Relevante para Órganos Jurisdiccionales':
        st.write('1.Nombre de la Norma: 2014-56-1-AC_V80\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2014-56-1-AC_V80.PDF\n Resumen: El Consejo de la Judicatura Federal emitió un acuerdo integral que describe las disposiciones administrativas para sus actividades, enfatizando la independencia, simplificando los procesos regulatorios y abordando diversos aspectos como los recursos humanos, la presupuestación, procedimientos de adquisición y gobernanza digital. El acuerdo contempla normas para el uso de firmas electrónicas, herramientas digitales y el manejo de archivos personales para servidores públicos, así como lineamientos para el uso de códigos QR, reserva fondos y procesos de adquisición. También detalla los procedimientos para administrar los fondos de ahorro, proporcionar servicios de cuidado infantil y apoyo de vivienda para los funcionarios públicos, al tiempo que enfatiza el cumplimiento de las leyes, regulaciones y el bienestar de los niños y servidores públicos. Además, el documento describe los procedimientos para arrendar, adquirir, administrar y enajenar propiedades inmobiliarias, la gestión financiera y los requisitos de presentación de informes dentro del Consejo de la Judicatura Federal. También discute acuerdos relacionados con la protección civil, los nombramientos judiciales y la reestructuración organizativa, con un enfoque en mejorar la transparencia, la rendición de cuentas y la eficiencia dentro del Consejo.\n\n2. Nombre de la Norma: 2020-12-0-AC_V07\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2020-12-0-AC_V07.PDF\n Resumen: El Acuerdo General 12/2020 del Pleno del Consejo de la Judicatura Federal en México se centra en regular el uso de archivos electrónicos y videoconferencias en materia judicial para mejorar el acceso a la justicia, particularmente durante la COVID-19 pandemia. Permite el uso de medios electrónicos en los procedimientos judiciales, describe las pautas para las notificaciones electrónicas, las firmas digitales y las videoconferencias, y enfatiza la importancia de mantener la integridad y la seguridad de archivos electrónicos. El documento tiene como objetivo modernizar y agilizar el proceso legal a través de medios digitales al tiempo que garantiza la seguridad y validez de los documentos electrónicos. También detalla los esfuerzos de modernización del Consejo de la Judicatura Federal, incluida la implementación de la tecnología de videoconferencia, y enfatiza la importancia de la seguridad, el cifrado y el respeto de los derechos de todas las partes involucradas en los procedimientos legales.\n\n3. Nombre de la Norma: 2018-28-2-AC_V03\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2018-28-2-AC_V03.PDF\n Resumen: El Consejo de la Judicatura Federal emitió un acuerdo general sobre la protección de datos personales, que describe los procedimientos para manejar las solicitudes y garantizar el cumplimiento de las leyes de protección de datos. Discute los principios para proteger los datos personales, el consentimiento para el procesamiento de datos y medidas para la seguridad de los datos. El documento también establece responsabilidades para los servidores públicos en el manejo de datos personales y describe los procedimientos para ejercer los derechos ARCO. Adicionalmente, detalla la implementación de la mano de obra reformas de justicia, incluida la creación de Tribunales Federales de Trabajo en varios estados de México.')
    else:
        '1. Nombre de la Norma: 2014-56-1-AC_V80\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2014-56-1-AC_V80.PDF\n Resumen: El Consejo de la Judicatura Federal emitió un acuerdo integral que describe las disposiciones administrativas para sus actividades, enfatizando la independencia, simplificando los procesos regulatorios y abordando diversos aspectos como los recursos humanos, la presupuestación, procedimientos de adquisición y gobernanza digital. El acuerdo contempla normas para el uso de firmas electrónicas, herramientas digitales y el manejo de archivos personales para servidores públicos, así como lineamientos para el uso de códigos QR, reserva fondos y procesos de adquisición. También detalla los procedimientos para administrar los fondos de ahorro, proporcionar servicios de cuidado infantil y apoyo de vivienda para los funcionarios públicos, al tiempo que enfatiza el cumplimiento de las leyes, regulaciones y el bienestar de los niños y servidores públicos. Además, el documento describe los procedimientos para arrendar, adquirir, administrar y enajenar propiedades inmobiliarias, la gestión financiera y los requisitos de presentación de informes dentro del Consejo de la Judicatura Federal. También discute acuerdos relacionados con la protección civil, los nombramientos judiciales y la reestructuración organizativa, con un enfoque en mejorar la transparencia, la rendición de cuentas y la eficiencia dentro del Consejo.\n\n2. Nombre de la Norma: 2018-45-1-AC_V12\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2018-45-1-AC_V12.PDF\n Resumen: El Acuerdo General del Pleno del Consejo de la Judicatura Federal establece disposiciones para las responsabilidades administrativas, el control financiero y la rendición de cuentas dentro del poder judicial. Describe la autoridad del Consejo para supervisar la judicial, emitir acuerdos generales, regular investigaciones y sanciones por responsabilidades administrativas, y enfatizar principios como la legalidad, la presunción de inocencia y el respeto a los derechos humanos. El documento detalla los procedimientos para investigar y sancionar las responsabilidades administrativas, incluida la creación de una Unidad de Investigación de Responsabilidades Administrativas, declaración de bienes e intereses, responsabilidades de los servidores públicos, imposición de sanciones, notificaciones, medidas cautelares, reembolso de ingresos perdidos, sanciones económicas, monitoreo de declaraciones de activos y manejo de procedimientos de responsabilidad administrativa. También enfatiza la transparencia, la rendición de cuentas y la importancia de incorporar la perspectiva de género en los casos de conducta sexual inapropiada o violencia de género.\n\n3. Nombre de la Norma: 2013-40-2-AC_V72\n Link: https://apps.cjf.gob.mx/normativa/Recursos/2013-40-2-AC_V72.PDF\n Resumen: El documento emitido por el Consejo de la Judicatura Federal en 2013 describe la organización, funciones y reformas dentro de la Judicatura Federal de México. Se centra en modernizar las normas institucionales, aclarar los procedimientos y garantizar la eficiencia y seguridad jurídica. El documento detalla las responsabilidades y funciones de varios departamentos dentro del Poder Judicial, incluida la supervisión de asuntos judiciales, la prestación de apoyo legal, la administración de finanzas, la implementación de políticas anticorrupción y coordinar los esfuerzos de comunicación. También describe los acuerdos realizados por el Consejo entre 2013 y 2023, abordando reformas y modificaciones en diversos aspectos de la organización y el funcionamiento del Consejo. '

En la app se vería así:

<div style="text-align: center;">
  <p><strong>Imagen 5: Ejemplo de clasificación</strong></p>
  <img src="imagenes/05_clasificacionA.png" width="70%">
  <p><em>Fuente: Elaboración propia</em></p>
</div>

<strong> <span style="font-size: 22px;"> Paso 8 Chatear con los documentos </span></strong>



Finalmente, para implementar la segunda funcionalidad, la de *chatear* con los documentos, ahora se utiliza el segundo *index*
Notece algo importante, a diferencia de otros *chatbots*, este, antes de "contestar", aal iguala que en clasificación, se conecta con nuestros vectores de pinecone, hace una busqueda, recupera los fragmentos relevantes, y los sutiliza como contexto para responder. 

Así pues, primero ponemos los datos básicos de esta parte de la app. 

<div style="text-align: center;">
  <p><strong>Imagen 6: Encabezados chat </strong></p>
  <img src="imagenes/06_encabezado_chat.png" width="70%">
  <p><em>Fuente: Elaboración propia</em></p>
</div>

Posteriormente inciamos el index, como en la clasificación. Pero esta vez, lo debemos de guardar como un vectorstore. Esto luce, bien, pero hace falta agregar el RAG, y los diversos *system prompts* que queremos que use el chatbot. Para ello, primero creamos un *eslabón* de la cadena, que contenga el prompt.

In [ ]:
#Primero cargamos las librerias para hacer el chat, y los system prompts
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

#Segundo definimos el systemprompt
system_prompt1 = (f"Eres un consultor experto en leyes. Por lo cual, vas a contestar las preguntas que se te hagan, de una manera clara y precisa. Para responder, te basarás sólo en los documentos que tengan el siguiente nombre {normas_chat}. Además, por favor consteta a los usuarios de manera amigable. Porbablemente te digan sus nombres. Su edad, a que se dedican. Datos por el estilo. ")

#Tercero, hacemos el promt template con el system prompt ya definido anteriormente 
prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt1), #lo que va entre " " es como el rol que ese espera que siga la instruccion que va del otro lado e la coma. en este sentido, le digo que el sistema debe adoptar ese systempropmt
     ("human", "{context}"),]) #aqui el papel de humano adoptara lo que sea que le pregunte

Después de ello, creamos una variable que contenga nuestro LLM, este será el segundo *eslabón*. El tercero será convertir el *vector store* en un *retriever* que nos de respuesta (tercer *slabón*). Finalmente, unimos toda la cadena. 

In [ ]:
#Por seguridad, omito la llamada al LLM.


#Transformamos el vectorstore en retriever


retriever = vector_store_chat.as_retriever()


#creamos el question answer chain y el rag chain

question_answer_chain = create_stuff_documents_chain(llm, prompt) #aqui recupro el promp y el llm que usare

rag_chain1 = create_retrieval_chain(retriever, question_answer_chain) #aqui recupero el rag con la variable retriever

Por último, hacemos el bot y lo integramos a la app. 


In [ ]:

def get_robot_response (user_message):
    response = rag_chain1.invoke({"input": user_message})
    return response ['answer']



def send_message():
    user_message = st.session_state.user_input 
    if user_message:
        bot_response = get_robot_response(user_message) 
        st.session_state.messages.append({"user":user_message, "bot": bot_response})
        st.session_state.user_input = "" 

#Lo implementamos


#Entrada de texto para el usuario

st.text_input ("Tu mensaje", key = "user_input", on_change=send_message) #key = nombre del ussairoo, que está en st.session_state.  on_change = me dice con que funcion de mensaje se va a usar el input


#Mostrar el chat

for message in st.session_state.messages: #itera sobre todos los mensajes guardados. Esto es lo que permite que se tenga memoria. 
    with st.chat_message("user"):
        st.markdown(message["user"])
    with st.chat_message("bot"):
        st.markdown (message["bot"])

En la aplicación se vería así:

<div style="text-align: center;">
  <p><strong>Imagen 7: Ejemplo de chat </strong></p>
  <img src="imagenes/07_ejemplo_chat.png" width="70%">
  <p><em>Fuente: Elaboración propia</em></p>
</div>

Por supuesto, falta ajustar las respuestas, y los *propmts* pero cumple el objetivo, el RAG funciona, y recuerda los mensajes anteriores. :3

<strong> <span style="font-size: 22px;"> Paso 9: Datos de salida </span></strong>

Finalmente, agregamos los datos de salida. 

In [ ]:
st.header ("Referencias")

st.markdown("Enlace al consejo de la judicatura: https://www.cjf.gob.mx/") #Poner un enlace
st.markdown("Enlace a la normativa aplicable: https://apps.cjf.gob.mx/normativa/Index") #Aqui poner los links al repositorio de normas.

st.markdown("Enlace CIDE: https://www.cide.edu/")

st.markdown("Cualquier duda, sugerencia, o error, por favor comunicarlo a jesus.villegas@alumnos.cide.edu")

st.markdown("Aplicación realizada por: Abraxalandro")

Y se vería así en la app:

<div style="text-align: center;">
  <p><strong>Imagen 8: Referencias </strong></p>
  <img src="imagenes/08_ref.png" width="70%">
  <p><em>Fuente: Elaboración propia</em></p>
</div>

<center> <strong> <span style="font-size: 27px;"> Conclusiones </span></strong> </center>

Las posibilidades que nos ofrece la inteligencia artificial y lo LLM's en particular son enormes. Aquí sólo se muestra una pequeña demo de cómo de una aplicación. Naturalmente, hay muchas mejoras posibles, como un mejor *prompt* o técnicas de RAG más avanzadas, o un mejor *frontend*. Sin embaego, eso será tarea de un desarrollo futuro. 

Lamento las ambigüedades y omisiones del código, pero nunca se es demasiado precavido en internet...

Si quieres probar el demo, puees ponerte en contacto conmigo al siguiente correo: java.law.1993@gmail.com